# 配置训练任务

上一节介绍了本章“三步闭环”的整体目标，本节聚焦三个核心问题：数据管线、训练配置与并行策略。

---

## 1. Wordle 数据集概览

我们使用的数据集 `willcb/V3-wordle` 中，每条样本记录了一局完整的 Wordle 游戏：

```json
{
  "prompt": [
    {"role": "system", "content": "Follow the Wordle instructions and required format."},
    {"role": "user", "content": "A secret 5-letter word has been chosen. You have 6 attempts."}
  ],
  "completion": [
    {"role": "assistant", "content": "<think>Test common letters.</think><guess>[crane]</guess>"},
    {"role": "user", "content": "CRANE -> X X G G G. You have 5 guesses left."},
    {"role": "assistant", "content": "<think>Keep A, N, E fixed.</think><guess>[plane]</guess>"}
  ],
  "answer": "plane",
  "reward": 1.0,
  "task": "wordle"
}
```


| 字段 | 用途 |
|------|------|
| `prompt` | 对话的起始消息段 |
| `completion` | 后续 assistant/user 多轮消息段 |
| `answer/reward/task` | 数据集元数据；当前 sample processor 不会将这些字段追加到消息中 |

当前 `process_wordle_sample` 执行 `list(prompt) + list(completion)`，训练对话由这两个字段共同组成；配置通过 `chat_processor` 字符串在运行时导入该函数。评测使用的秘密词来自独立的固定 Wordle eval 任务集。

---

## 2. 训练配置总览

下面按 `config_registry.py` 的构造顺序展示 `sft_qwen3_1_7b_wordle()`。它首先从 `_qwen3_1_7b_base()` 和 `sft_default_config()` 获取默认值，再通过 `dataclasses.replace()` 仅覆盖本课程相关的字段；未列出的字段均继承默认配置。

```python
base = sft_default_config(_qwen3_1_7b_base())
return replace(
    base,
    optimizer=replace(base.optimizer, lr=1e-5),
    lr_scheduler=replace(
        base.lr_scheduler, warmup_steps=0, decay_ratio=1.0, min_lr_factor=1.0
    ),
    training=replace(
        base.training, local_batch_size=2, global_batch_size=64,
        seq_len=1024, max_norm=1.0, steps=20
    ),
    dataloader=ChatDataLoaderConfig(
        dataset_path="willcb/V3-wordle",
        load_dataset_kwargs={"split": "train"},
        chat_processor="torchtitan_npu.hf_datasets.chat_processors.process_wordle_sample",
    ),
    checkpoint=replace(
        base.checkpoint, folder="checkpoint_wordle_sft",
        initial_load_in_hf=True, initial_load_path="./assets/hf/Qwen3-1.7B"
    ),
    activation_checkpoint=replace(base.activation_checkpoint, mode="selective"),
    profiling=replace(
        base.profiling, profile_ranks=[0], profile_step_start=6,
        profile_step_end=7, profile_with_memory=True
    ),
)
```

下面逐项展开。

---

## 3. 数据加载

```python
dataloader=ChatDataLoaderConfig(
    dataset_path="willcb/V3-wordle",
    load_dataset_kwargs={"split": "train"},
    chat_processor="torchtitan_npu.hf_datasets.chat_processors.process_wordle_sample",
),
```

一条原始样本需经过四个步骤才能成为模型可用的 batch。先明确每一步解决的问题：`chat_processor` 选择并整理对话消息；chat template 与 tokenizer 将消息转换为 token；label 构造决定哪些位置计算 loss；packing 将若干短对话装入固定长度的序列。

```text
raw sample（prompt/completion）
  → chat_processor：消息列表
  → chat template + tokenizer：full_text / full_tokens
  → shifted input_ids / labels + assistant mask
  → greedy packing：固定 seq_len 的 packed sequence
```

### 3.1 Sample Processor

`process_wordle_sample` 接收一条数据集记录，将 `prompt` 和 `completion` 连接成按时间排序的消息列表，并把列表形式的 `content` 统一为字符串。其输出仍是 `[{"role": ..., "content": ...}, ...]`，此时尚未生成 token ID 或 labels。

### 3.2 Chat Template

模型最终接收的是 token 序列。chat template 先将消息列表序列化为带角色边界的文本；Qwen3 使用 `<|im_start|>` 和 `<|im_end|>` 标记每条消息，例如：

```text
<|im_start|>user
法国的首都是哪里？<|im_end|>
<|im_start|>assistant
巴黎。<|im_end|>
```

不同模型使用的特殊标记可能不同，但作用一致：保留消息角色与边界。

### 3.3 Token 与 Labels

tokenizer 按词表将文本转换为 `full_tokens`。ChatDataset 使用 `full_tokens[:-1]` 作为 `input_ids`，使用 `full_tokens[1:]` 作为 next-token labels，然后将非 assistant 目标位置替换为 `-100`。因此，`chat_processor` 只负责生成消息，ChatDataset 才生成 token ID 和 labels。

### 3.4 长度检查与 Packing

若某条样本 tokenization 后长度超过 `seq_len=1024`，ChatDataset 会丢弃该样本。长度合格的样本按顺序加入 packing buffer；当下一条样本无法放入当前 buffer 时，框架用 padding 填满 buffer，输出一条长度为 1024 的 packed sequence，然后从新的 buffer 继续。EOS 标记对话边界，position 在每条对话开始时重新从 0 计数。具体实现可参考 [TorchTitan ChatDataset 源码](https://github.com/pytorch/torchtitan/blob/ac13e536c84e7f6647b14fa9375c3c8a8a2b8578/torchtitan/hf_datasets/text_datasets.py)。

---

## 4. 训练运行配置

下面展开训练相关的配置项。

### 4.1 梯度累积

```python
training=TrainingConfig(
    local_batch_size=2,      # 每个 rank 每个 microbatch 包含 2 条 packed sequence
    global_batch_size=64,    # 每次 optimizer step 的全局 batch elements
    ...
)
```

先区分四个关键概念：

- **原始样本**：数据集中的一局 Wordle 对话，长度可长可短。
- **packed sequence**：DataLoader 输出的一个固定长度训练元素，本课程 shape 为 `[1024]`，可能包含多局较短对话。
- **microbatch**：一次 forward/backward 中每个 rank 取到的 packed sequences；`local_batch_size=2`，因此 tensor 的前两维为 `[2, 1024]`。
- **optimizer step**：累积足够 microbatches 的梯度后，优化器更新一次参数。

本课程启动两个 rank，DP batch degree 为 2。每个 rank 每个 microbatch 读取 2 条 sequence，因此一次 microbatch 全局处理 `2 × 2 = 4` 条。每个 rank 累积 16 次 forward/backward，即各处理 32 条 packed sequences；两个 rank 合计 64 条后执行一次 optimizer step。

```text
rank 0: [2, 1024] ─┐
                    ├─ microbatch #1：全局 4 条 packed sequences → 累积梯度
rank 1: [2, 1024] ─┘
...
microbatch #16：全局累积 64 条 packed sequences → optimizer.step()
```

公式：`梯度累积次数 = global_batch_size ÷ (local_batch_size × data_parallel_batch_degree)`

本课程：`64 ÷ (2 × 2) = 16`。TorchTitan 将 `global_batch_size` 作为启用梯度累积的配置项，详见[官方 README](https://github.com/pytorch/torchtitan#key-features-available)。训练初始化日志应打印 `local batch size 2, global batch size 64, gradient accumulation steps 16`，以确认最终解析值。

### 4.2 其他训练参数

| 参数 | 值 | 说明 |
|------|-----|------|
| `seq_len` | 1024 | tokenized raw sample 超长时整条丢弃；较短样本则进行 greedy packing 和 padding |
| `max_norm` | 1.0 | 计算全模型总梯度范数后执行裁剪；日志记录的是裁剪前 `total_norm`，因此可能大于 1.0 |
| `steps` | 20 | optimizer step 数；是否学会格式或策略需用独立评测验证 |

### 4.3 学习率

```python
lr_scheduler=LRSchedulersContainer.Config(
    warmup_steps=0,        # 不进行 warmup
    decay_ratio=1.0,       # 调度区间覆盖全部训练步
    decay_type="cosine",   # 余弦调度器
    min_lr_factor=1.0,     # 终点与起点相同，因此 lr 保持不变
)
```

学习率控制参数更新的步幅。`lr=1e-5` 是 optimizer 的峰值学习率。TorchTitan 的调度器按 `warmup → stable → decay` 三个阶段计算：warmup 从较小步幅线性升至峰值，stable 保持峰值，decay 再按 linear、sqrt 或 cosine 下降。

当前配置中 `warmup_steps=0`、`decay_ratio=1.0`、`min_lr_factor=1.0` 使余弦终点仍为 `1e-5`，因此这 20 步实际为常数学习率。`min_lr_factor` 控制终点与初始学习率的比例；例如设为 `0.1`，终点即为 `1e-6`。

实际训练通常先用 warmup 降低初始更新过大的风险，再在较长区间使用 cosine decay，使后期步幅逐渐变小。TorchTitan 配置文档建议 warmup 约占训练步数的五分之一；其 Qwen3-8B SFT 示例使用了 `warmup_steps=15`、`decay_ratio=0.9`、`min_lr_factor=0.1`（见[固定上游配置](https://github.com/pytorch/torchtitan/blob/ac13e536c84e7f6647b14fa9375c3c8a8a2b8578/torchtitan/models/qwen3/config_registry.py#L267-L281)）。实际值需结合有效 batch、总 token 数、模型规模和验证集曲线进行调整。

---

## 5. 多卡训练

本课程主线使用两个 NPU 进程。并行维度的选择需同时满足模型可放置性、world size 整除性和实际吞吐收益。

### 5.1 并行配置选项

框架提供七种并行维度，对应七个配置参数：

```python
parallelism=ParallelismConfig(
    data_parallel_replicate_degree=1,    # DP 副本（DDP）
    data_parallel_shard_degree=-1,       # DP 分片（FSDP），-1=自动
    tensor_parallel_degree=1,            # TP 张量并行
    pipeline_parallel_degree=1,          # PP 流水线并行
    expert_parallel_degree=1,            # EP Expert并行
    expert_tensor_parallel_degree=1,     # ETP Expert+张量并行
    context_parallel_degree=1,           # CP 上下文并行
)
```

各策略的定位如下：

| 并行策略 | 配置项 | 思路 | 适用场景 |
|----------|--------|------|-----------|
| **DP (FSDP)** | `data_parallel_shard_degree` | 不同 rank 处理不同数据，并分片参数、梯度和优化器状态 | 需降低每 rank 模型状态显存时 |
| DP (DDP) | `data_parallel_replicate_degree` | 不同 rank 处理不同数据，每个副本保留完整模型状态 | 完整模型可放入单 rank，且复制能带来吞吐收益时 |
| **TP** | `tensor_parallel_degree` | 将层内张量运算切分到多个 rank | 单层无法放下，或实测层内切分有收益时 |
| **PP** | `pipeline_parallel_degree` | 将连续层分到不同 stage | 需按层切分模型，且流水调度开销可接受时 |
| **CP** | `context_parallel_degree` | 沿序列维度切分计算 | 长上下文导致激活或注意力开销成为瓶颈时 |
| **EP** | `expert_parallel_degree` | MoE 的 expert 分配到不同卡 | 模型包含 MoE 结构（如 DeepSeek） |
| **ETP** | `expert_tensor_parallel_degree` | EP + TP 组合 | MoE 且 expert 也很大时 |

### 5.2 配置选择

针对 Qwen3-1.7B + Wordle SFT 任务：

| 条件 | 判断 | 结论 |
|------|------|------|
| 当前单层可放入单个 NPU | 暂不需要层内切分 | TP=1 |
| 当前模型可由 FSDP2 处理 | 暂不需要按层划分 stage | PP=1 |
| seq_len=1024，当前不使用上下文切分 | 保持序列完整 | CP=1 |
| 当前模型为 dense Transformer | 无 expert 维度 | EP=1, ETP=1 |
| 两个训练进程参与数据并行 | 让自动分片占用剩余并行维度 | DP shard=-1 |

当前配方仅让 `data_parallel_shard_degree` 自动解析，其余维度保持 1。`data_parallel_shard_degree` 即 FSDP 的权重分片度；TorchTitan 构造 dense mesh 时有效 FSDP degree 为 `dp_shard × CP`，本课程 `CP=1`，因此 FSDP degree 就是 `dp_shard`。

`data_parallel_shard_degree=-1` 为自动模式：
- 计算方式：`dp_shard = world_size ÷ (dp_replicate × TP × PP × CP)`；
- 本课程 `world_size=2`，其余相关维度均为 1，因此 `dp_shard=2`；
- 若同样用两个 rank 但设置 `TP=2`，剩余 `dp_shard` 只能是 1。

> `-1` 表示自动计算，不等于设备数。配置生效的条件是各并行维度的乘积等于 `world_size`；实现细节见 [TorchTitan `ParallelDims`](https://github.com/pytorch/torchtitan/blob/ac13e536c84e7f6647b14fa9375c3c8a8a2b8578/torchtitan/distributed/parallel_dims.py#L67-L73)。

### 5.3 并行策略选择指南

```text
完整模型状态能否放入单 rank？          → 比较 DDP 与 FSDP
单层是否能放入单 rank，TP 是否实测受益？ → 决定是否启用 TP
是否需要按层切分，流水气泡是否可接受？    → 决定是否启用 PP
长上下文是否成为主要瓶颈？              → 决定是否启用 CP
模型是否包含可并行的 MoE experts？       → 决定是否启用 EP/ETP
最后检查所有并行维度乘积等于 world_size，并用基准测试确认收益。
```

---

## 6. Activation Checkpoint

此外，Activation Checkpoint 可进一步节省显存：

```python
activation_checkpoint=ActivationCheckpointConfig(mode="selective")
```

前向传播会产生反向计算所需的激活。保存更多激活会占用更多显存；重算部分激活则会增加计算开销。

TorchTitan 当前的 `selective` 模式按算子决定保存哪些输出、反向时重算哪些输出，因此不等同于仅对某几层做 checkpoint。显存节省和额外耗时随模型、序列长度和算子实现变化，请以实际 profile 为准。

---

## 小结

本节走完了从数据到配置的完整链路：

- **数据**：`chat_processor` 将原始记录整理为消息列表；ChatDataset 再应用 chat template、构造 shifted labels 和 assistant mask，并将短样本打包到固定长度。
- **训练**：`global_batch_size = local_batch_size × DP batch degree × 梯度累积次数`；本课程为 `64 = 2 × 2 × 16`。
- **学习率**：`min_lr_factor=1.0` 使调度起点与终点相同，因此当前配方保持 `1e-5` 常数学习率。
- **多卡**：本课程两个 rank 解析为 `dp_shard=2`；是否增加 TP/PP/CP/EP 取决于放置约束、整除关系和实测收益。
- **省显存**：selective activation checkpoint 按算子策略权衡保存与重算。

下一节将使用此配置启动训练。

---

## 练习

1. （判断题）`global_batch_size=64`、`local_batch_size=2`、DP batch degree=2 时，梯度累积次数为 16。

2. （单选题）`chat_processor` 的直接输出是什么？  
   A. 按时间排序的 role/content 消息列表  
   B. 已经分片的 parameter DTensor  
   C. 只有 assistant token 的 labels  
   D. 评测 reward 向量  

3. （单选题）要将当前 20 步常数学习率改为有 warmup 和后期余弦衰减的配置，哪一项合适？  
   A. `warmup_steps=0, min_lr_factor=1.0`  
   B. `warmup_steps>0, decay_type="cosine", min_lr_factor=0.1`  
   C. `warmup_steps=-1, decay_type="linear"`  
   D. 只增大 `local_batch_size`，不改 scheduler  

4. （多选题）选择 TP degree 时应同时检查哪些条件？  
   A. 单层参数和中间张量能否按目标维度放置  
   B. world size 和 attention head 等维度是否整除  
   C. 通信开销后的实测吞吐  
   D. 只看模型总参数量，不需要运行 profile

In [ ]:
!cat ./answer/03.02_answer.txt